# 02. YOLO Baseline + YOLO/OpenCV 이미지 증강 실험 + 최적 조합 탐색

이 노트북은 **1번 노트북이 만든 `../../data/processed`만 읽어서** 모델 실험을 수행합니다.

목표는 단순히 YOLO 한 번 학습하는 것이 아니라 다음 질문에 답하는 것입니다.

> “우리 재활용품 데이터에서는 어떤 이미지 증강이 실제 Validation 성능을 높이는가?”

이를 위해 각 기법을 하나씩 분리해서 실험하고, 이후 여러 기법을 조합한 실험까지 수행합니다.

## 전체 흐름

1. processed 데이터 다시 확인
2. GPU / YOLO 버전 / 기본 augmentation 설정 확인
3. 평가 지표 이해
4. 증강이 필요한 이유와 주의점 이해
5. YOLO 내장 증강 기법별 원리 학습
6. OpenCV 증강 기법별 원리 학습
7. OpenCV 증강 결과 bbox 시각 검증
8. 무증강 control 학습
9. Ultralytics 기본 baseline 학습
10. YOLO 단일 증강 실험
11. YOLO 조합 실험
12. OpenCV 단일 증강 실험
13. OpenCV 조합 실험
14. OpenCV + YOLO hybrid 실험
15. 모든 실험 Precision / Recall / mAP50 / mAP50-95 비교
16. 상위 조합 multi-seed 재검증
17. 최종 best augmentation 선택
18. Baseline vs Best 클래스별 AP 비교
19. PR curve / confusion matrix / 실제 예측 결과 확인

---

## 매우 중요한 원칙

### Validation에는 증강을 적용하지 않습니다.

Validation은 모델이 학습 중 보지 않은 “고정 시험지” 역할을 해야 합니다.
Validation까지 랜덤하게 바꾸면 실험마다 시험 문제가 달라져 비교가 불공정해집니다.

### 모든 실험은 새 pretrained model에서 시작합니다.

A 실험을 학습한 뒤 그 weight에서 B 실험을 이어서 학습하면 B가 더 많은 학습을 받은 셈입니다.
따라서 매 실험마다 `YOLO(MODEL_NAME)`을 새로 생성합니다.

### 최적이라는 말의 의미

이 노트북이 찾는 “최적”은 **여기에서 정의한 후보 증강 조합 중 Validation mAP50-95가 가장 안정적으로 높은 조합**입니다.
최종 서비스 성능을 확정하려면 나중에 별도의 Test set으로 한 번 더 평가해야 합니다.

# 0. 현재 Ultralytics 기준으로 사용하는 기능

이 노트북은 현재 Ultralytics YOLO의 Detection 학습/검증 API 흐름을 기준으로 작성했습니다.

주요 내장 증강:

- HSV color jitter: `hsv_h`, `hsv_s`, `hsv_v`
- 가로/세로 flip: `fliplr`, `flipud`
- 회전/이동/확대축소/shear/perspective: `degrees`, `translate`, `scale`, `shear`, `perspective`
- Mosaic: `mosaic`
- MixUp: `mixup`
- CutMix: `cutmix`

현재 공식 문서에서 Copy-Paste는 polygon label이 필요한 segment/OBB 계열에 적용되는 기법이므로,
**bbox-only Detection processed 데이터에는 Copy-Paste를 실험 후보로 넣지 않습니다.**

참고 문서:

- https://docs.ultralytics.com/modes/train/
- https://docs.ultralytics.com/modes/val/
- https://docs.ultralytics.com/guides/yolo-data-augmentation/
- https://docs.ultralytics.com/usage/cfg/

설치 과정은 이 노트북에 포함하지 않습니다.

# 1. 라이브러리 불러오기

- `ultralytics.YOLO`: 모델 학습과 검증
- `torch`: GPU 확인 및 메모리 정리
- `cv2`: OpenCV 오프라인 증강
- `pandas`: 실험 결과표
- `matplotlib`: 모든 실험 비교 그래프
- `yaml`: 실험별 dataset yaml 생성

OpenCV 증강은 **이미지뿐 아니라 bbox도 같이 변환**해야 합니다.
기하학적 변환에서 bbox를 그대로 두면 완전히 잘못된 정답으로 학습하게 되므로 이 부분을 특히 엄격하게 처리합니다.

In [ ]:
from __future__ import annotations

import json
import math
import time
import random
import hashlib
import shutil
import gc
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import yaml
import torch

from ultralytics import YOLO

try:
    import ultralytics
    ULTRALYTICS_VERSION = ultralytics.__version__
except Exception:
    ULTRALYTICS_VERSION = "unknown"

try:
    from ultralytics.cfg import DEFAULT_CFG_DICT
except Exception:
    DEFAULT_CFG_DICT = {}

from IPython.display import display, Image as IPImage

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 500)


def set_korean_font():
    candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
    installed = {font.name for font in fm.fontManager.ttflist}

    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break

    plt.rcParams["axes.unicode_minus"] = False


set_korean_font()

print("Ultralytics:", ULTRALYTICS_VERSION)
print("PyTorch    :", torch.__version__)
print("OpenCV     :", cv2.__version__)

# 2. 실험 경로와 공통 설정

이번 프로젝트에서는 다음 경로 규칙을 사용합니다.

```text
현재 PROJECT_ROOT
│
├─ ../../data/sample/train9val2.zip
│  └─ 1번 노트북이 읽는 원본 ZIP
│
├─ ../../data/processed/
│  └─ 1번 노트북이 만든 YOLO 학습 데이터
│
└─ ../models/yolo/01_experiment_augmentation/
   ├─ runs/
   │  └─ YOLO 학습 checkpoint와 Ultralytics run 산출물
   ├─ opencv_datasets/
   │  └─ OpenCV 정적 증강 실험용 임시 데이터
   └─ report/
      ├─ preprocess/
      │  └─ 1번 노트북의 EDA/전처리 CSV
      ├─ summary/
      │  └─ 모든 실험 성능표와 비교 결과
      ├─ per_class/
      │  └─ 실험별 클래스 성능
      └─ final_best/
         └─ 최종 best 모델 평가 시각화
```

즉, **사람이 확인할 보고서와 성능 비교 자료는 모두 요청한 `report/` 아래에 모으고**,
학습 checkpoint나 OpenCV 임시 데이터처럼 용량이 큰 산출물은 그 상위 실험 폴더에 분리합니다.

## 가장 자주 바꾸는 값

- `MODEL_NAME`: pretrained 모델
- `EPOCHS`: 모든 본 실험의 최대 epoch
- `IMGSZ`: 입력 크기
- `BATCH`: batch size
- `PATIENCE`: early stopping

경로는 아래 코드 셀에 프로젝트 상대경로로 고정해 두었습니다.

## 실행 스위치

- `RUN_EXPERIMENTS=True`: 전체 후보 실험 실행
- `RUN_MULTI_SEED=True`: 상위 3개를 여러 seed로 다시 검증
- `SKIP_COMPLETED=True`: 이미 성공한 실험은 다시 돌리지 않음
- `SMOKE_TEST=True`: 파이프라인만 빠르게 확인할 때 사용

현재 기본값은 **실제 전체 실험용**입니다.
처음 코드 오류 여부만 확인하고 싶다면 `SMOKE_TEST=True`로 바꾸세요.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

# ------------------------------------------------------------
# 1) 1번 노트북이 만든 processed 데이터
# ------------------------------------------------------------
PROCESSED_DIR = (
    PROJECT_ROOT / "../../data/processed"
).resolve()

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"processed 폴더가 없습니다: {PROCESSED_DIR}\n"
        "먼저 01_recycling_eda_preprocess_build_processed.ipynb를 실행하세요."
    )

DATA_YAML = PROCESSED_DIR / "data.yaml"

# ------------------------------------------------------------
# 2) 모델/실험 산출물 위치
# ------------------------------------------------------------
EXPERIMENT_ROOT = (
    PROJECT_ROOT / "../models/yolo/01_experiment_augmentation"
).resolve()

RUNS_DIR = EXPERIMENT_ROOT / "runs"
CV_CACHE_DIR = EXPERIMENT_ROOT / "opencv_datasets"

# ------------------------------------------------------------
# 3) 사람이 확인할 보고서 위치
# ------------------------------------------------------------
REPORT_ROOT = EXPERIMENT_ROOT / "report"
REPORT_SOURCE_DIR = REPORT_ROOT / "preprocess"
SUMMARY_DIR = REPORT_ROOT / "summary"
PER_CLASS_DIR = REPORT_ROOT / "per_class"
FINAL_DIR = REPORT_ROOT / "final_best"

for path in [
    EXPERIMENT_ROOT,
    RUNS_DIR,
    CV_CACHE_DIR,
    REPORT_ROOT,
    REPORT_SOURCE_DIR,
    SUMMARY_DIR,
    PER_CLASS_DIR,
    FINAL_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "yolo26n.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 8
PATIENCE = 12
WORKERS = 0
SEED = 42

RUN_EXPERIMENTS = True
RUN_MULTI_SEED = True
SKIP_COMPLETED = True
SMOKE_TEST = False

# OpenCV static augmentation은 원본과 같은 데이터 개수를 유지합니다.
CV_APPLY_PROBABILITY = 0.80
CV_OUTPUT_JPEG_QUALITY = 95
CLEANUP_CV_DATASET_AFTER_RUN = True

if SMOKE_TEST:
    EPOCHS = 2
    PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("PROCESSED_DIR  :", PROCESSED_DIR)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("RUNS_DIR       :", RUNS_DIR)
print("REPORT_ROOT    :", REPORT_ROOT)
print("SUMMARY_DIR    :", SUMMARY_DIR)
print("MODEL_NAME     :", MODEL_NAME)
print("EPOCHS         :", EPOCHS)
print("IMGSZ          :", IMGSZ)
print("BATCH          :", BATCH)
print("SMOKE_TEST     :", SMOKE_TEST)


# 3. GPU / Device 확인

CUDA GPU가 있으면 GPU 0을 사용합니다.
Apple Silicon에서는 MPS, 둘 다 없으면 CPU를 사용합니다.

6GB 안팎 VRAM에서 `batch=8`은 비교적 보수적인 출발값입니다.
CUDA out-of-memory가 발생하면 **다른 실험 조건은 그대로 두고 BATCH만 4 또는 2로 낮추세요.**

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM: {total_vram:.2f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple MPS 사용")
else:
    DEVICE = "cpu"
    print("CPU 사용")

print("DEVICE =", DEVICE)

# 4. processed `data.yaml`을 현재 경로에 맞게 보정

1번 노트북에서 만든 `../../data/processed` 폴더를 다른 PC나 다른 경로로 복사하면 `data.yaml`의 절대 `path`가 과거 위치를 가리킬 수 있습니다.

따라서 이 노트북은 원본 `data.yaml`을 읽고 `path`만 **현재 `PROCESSED_DIR`로 갱신한 runtime YAML**을 따로 만듭니다.
원본 파일은 수정하지 않습니다.

In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(DATA_YAML)

with open(DATA_YAML, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

dataset_config["path"] = str(PROCESSED_DIR.resolve())

RUNTIME_DATA_YAML = SUMMARY_DIR / "runtime_processed.yaml"

with open(RUNTIME_DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dataset_config, file, allow_unicode=True, sort_keys=False)

names_raw = dataset_config["names"]

if isinstance(names_raw, dict):
    CLASS_NAMES = {int(key): str(value) for key, value in names_raw.items()}
else:
    CLASS_NAMES = {index: str(value) for index, value in enumerate(names_raw)}

NUM_CLASSES = len(CLASS_NAMES)

print(RUNTIME_DATA_YAML.read_text(encoding="utf-8")[:5000])
print("클래스 수:", NUM_CLASSES)

# 5. 1번 노트북의 품질 보고서 읽기

모델 성능 숫자를 보기 전에 학습 데이터 규모를 항상 같이 확인해야 합니다.

특히 현재 Validation에는 객체가 없는 클래스가 존재할 수 있습니다.
그 클래스의 AP는 이 Validation set으로는 신뢰성 있게 평가할 수 없습니다.

In [ ]:
CLASS_SUPPORT_CSV = REPORT_SOURCE_DIR / "class_support_processed.csv"
PREPROCESS_SUMMARY_CSV = REPORT_SOURCE_DIR / "preprocess_summary.csv"

if CLASS_SUPPORT_CSV.exists():
    class_support_df = pd.read_csv(CLASS_SUPPORT_CSV)
    display(class_support_df)

    no_val_classes = class_support_df[class_support_df["val"].eq(0)]
    print("Validation object가 0인 클래스:", len(no_val_classes))
    if len(no_val_classes):
        display(no_val_classes[["class_name", "train", "val"]])
else:
    class_support_df = pd.DataFrame()
    print("class_support_processed.csv를 찾지 못했습니다.")

if PREPROCESS_SUMMARY_CSV.exists():
    display(pd.read_csv(PREPROCESS_SUMMARY_CSV))

# 6. 학습 전 processed 빠른 재검증

1번 노트북에서 이미 검사했지만, 모델 실험을 시작하는 시점에 다음을 다시 확인합니다.

- Train/Val 이미지 폴더 존재
- Train/Val label 폴더 존재
- 이미지 수와 label 수가 같은지
- Validation이 실수로 비어 있지 않은지

In [ ]:
TRAIN_IMAGE_DIR = PROCESSED_DIR / "images" / "train"
VAL_IMAGE_DIR = PROCESSED_DIR / "images" / "val"
TRAIN_LABEL_DIR = PROCESSED_DIR / "labels" / "train"
VAL_LABEL_DIR = PROCESSED_DIR / "labels" / "val"

for path in [TRAIN_IMAGE_DIR, VAL_IMAGE_DIR, TRAIN_LABEL_DIR, VAL_LABEL_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)

train_images = sorted(path for path in TRAIN_IMAGE_DIR.iterdir() if path.is_file())
val_images = sorted(path for path in VAL_IMAGE_DIR.iterdir() if path.is_file())
train_labels = sorted(TRAIN_LABEL_DIR.glob("*.txt"))
val_labels = sorted(VAL_LABEL_DIR.glob("*.txt"))

print("Train images:", len(train_images))
print("Train labels:", len(train_labels))
print("Val images  :", len(val_images))
print("Val labels  :", len(val_labels))

if len(train_images) != len(train_labels):
    raise ValueError("Train image와 label 개수가 다릅니다.")

if len(val_images) != len(val_labels):
    raise ValueError("Validation image와 label 개수가 다릅니다.")

if len(val_images) == 0:
    raise ValueError("Validation 이미지가 없습니다.")

print("Processed quick check: PASSED")

# 7. 성능지표를 먼저 이해하기

증강 실험에서 숫자를 많이 보게 되므로 지표 의미를 먼저 이해하고 시작합니다.

## Precision

모델이 “객체가 있다”고 예측한 것 중 실제 정답인 비율입니다.

```text
Precision = TP / (TP + FP)
```

Precision이 낮으면 존재하지 않는 객체를 많이 검출하는 경향이 있습니다.

## Recall

실제 존재하는 객체 중 모델이 찾아낸 비율입니다.

```text
Recall = TP / (TP + FN)
```

우리 서비스처럼 사진 속 후보 객체를 먼저 보여주는 구조에서는 실제 물체를 놓치지 않는 Recall도 중요합니다.

## mAP50

IoU 0.50 이상을 맞는 bbox로 보는 비교적 관대한 AP 평균입니다.

## mAP50-95

IoU 0.50, 0.55, ..., 0.95처럼 더 엄격한 여러 기준에서 AP를 평균합니다.

**이 노트북에서는 mAP50-95를 1차 순위 지표로 사용**하고 Precision, Recall, mAP50을 함께 봅니다.

# 8. 왜 이미지 증강을 실험하는가?

현재 학습 이미지는 클래스당 수십~수백 장이 아니라 비교적 작은 규모입니다.
작은 데이터에서는 모델이 “프라이팬이라는 개념”보다 특정 배경, 특정 밝기, 특정 촬영 각도를 외워버릴 위험이 있습니다.

이미지 증강은 같은 정답을 유지하면서 사진을 조금씩 바꿔 모델에게 다음을 가르치는 방법입니다.

> “모양의 핵심 특징은 유지되지만 촬영 조건은 달라질 수 있다.”

하지만 증강은 강할수록 무조건 좋은 것이 아닙니다.
현실에서 거의 나오지 않는 왜곡을 지나치게 만들면 오히려 학습을 방해할 수 있습니다.

그래서 **한 기법씩 분리해서 성능을 측정하고, 도움이 된 기법들을 조합**합니다.

# 9. YOLO 온라인 증강 vs OpenCV 오프라인 증강

## YOLO 온라인 증강

학습 DataLoader가 이미지를 불러올 때마다 랜덤 변환을 적용합니다.
따라서 같은 원본 이미지라도 epoch마다 조금 다른 모습으로 모델에 들어갈 수 있습니다.

장점:

- 매 epoch 새로운 변형을 볼 수 있음
- 별도 이미지 파일을 저장할 필요가 없음
- bbox 변환을 YOLO 내부가 처리

## OpenCV 오프라인 증강

학습 전에 우리가 직접 변형 이미지를 파일로 만들어 둡니다.

장점:

- 정확히 어떤 픽셀 변환이 일어났는지 직접 제어 가능
- 모바일 촬영 품질 저하, JPEG artifact, motion blur 같은 커스텀 변형을 쉽게 실험 가능
- bbox 변환 원리를 직접 확인 가능

단점:

- 한번 만든 증강본은 고정됨
- 기하 변환 시 bbox도 직접 안전하게 바꿔야 함

### 공정 비교를 위한 이 노트북의 정책

OpenCV 실험은 기본적으로 **원본과 같은 장수의 static train dataset**을 만듭니다.
각 이미지에 80% 확률로 해당 변형을 적용하고 20%는 원본을 그대로 둡니다.
따라서 “이미지 수를 두 배로 늘려서 학습 step이 많아진 효과”가 증강 효과와 섞이는 것을 줄입니다.

# 10. YOLO 기법 ① HSV Color Jitter

### 무엇을 바꾸나?

- `hsv_h`: Hue, 색조
- `hsv_s`: Saturation, 채도
- `hsv_v`: Value, 밝기

### 원리

RGB를 직접 흔드는 대신 HSV 색공간의 요소를 랜덤하게 바꿉니다.
같은 프라이팬도 카메라, 조명, 화이트밸런스에 따라 색이 조금 다르게 찍힐 수 있다는 상황을 모사합니다.

### 기대 효과

- 조명과 색상 변화에 덜 민감해짐
- 특정 색을 클래스 정답처럼 외우는 현상 감소

### 위험

너무 강하면 실제 물체 색과 완전히 다른 비현실적 이미지가 되어 오히려 성능이 떨어질 수 있습니다.

# 11. YOLO 기법 ② Horizontal Flip

`fliplr=0.5`라면 약 50% 확률로 이미지를 좌우 반전합니다.

### 기대 효과

물체가 이미지 왼쪽에 있든 오른쪽에 있든 같은 물체라는 것을 학습합니다.

### 왜 Vertical Flip은 기본 실험에서 제외하나?

상하 반전은 실제 사용자가 휴대폰으로 찍는 방향과 지나치게 다른 경우가 많습니다.
특히 병, 가구, 전자제품처럼 위아래 방향이 의미 있는 물체에서는 비현실적인 학습 샘플을 만들 수 있습니다.

# 12. YOLO 기법 ③ Rotation

`degrees`는 이미지를 일정 각도 범위에서 랜덤 회전합니다.

### 기대 효과

사용자가 카메라를 약간 기울여 찍거나 물건이 비스듬히 놓인 상황에 강해질 수 있습니다.

### 위험

너무 크게 회전하면 이미지 모서리에 큰 빈 영역이 생기거나 객체 일부가 잘릴 수 있습니다.
그래서 여기서는 비교적 약한 ±10° 수준을 실험합니다.

# 13. YOLO 기법 ④ Translation

`translate`는 이미지를 좌우/상하로 이동합니다.

### 기대 효과

학습 이미지에서 객체가 항상 중앙에 있는 경우 생기는 위치 편향을 줄일 수 있습니다.
실제 서비스에서는 사용자가 물체를 정확히 중앙에 두지 않을 수 있습니다.

# 14. YOLO 기법 ⑤ Scale

`scale`은 이미지를 확대/축소합니다.

### 기대 효과

카메라가 물체에 가까운 사진과 먼 사진을 모두 인식하는 데 도움을 줄 수 있습니다.

### 주의

너무 축소하면 작은 객체가 더 작아져 특징이 사라질 수 있고,
너무 확대하면 객체 일부가 프레임 밖으로 잘릴 수 있습니다.

# 15. YOLO 기법 ⑥ Shear

Shear는 직사각형을 평행사변형처럼 기울이는 변환입니다.

### 기대 효과

정면이 아니라 비스듬한 각도에서 본 물체의 형태 변화에 대한 견고성을 높일 수 있습니다.

### 주의

현실 카메라 왜곡보다 지나치게 강하면 물체 모양을 부자연스럽게 만들 수 있으므로 약한 값만 사용합니다.

# 16. YOLO 기법 ⑦ Perspective

Perspective는 원근 왜곡을 추가합니다.

예를 들어 직사각형 물체를 비스듬한 방향에서 찍으면 가까운 쪽이 크게, 먼 쪽이 작게 보입니다.

### 기대 효과

다양한 카메라 각도에 강해질 수 있습니다.

### 위험

값이 너무 크면 실제 촬영보다 과도하게 찌그러집니다.
그래서 작은 값부터 실험합니다.

# 17. YOLO 기법 ⑧ Mosaic

Mosaic는 여러 학습 이미지를 하나의 큰 학습 샘플로 합칩니다.

### 기대 효과

- 한 장에서 여러 객체/배경을 동시에 학습
- 작은 객체 학습에 도움을 줄 수 있음
- 다양한 위치/크기의 객체를 경험

### 위험

우리 실제 서비스 입력은 일반적으로 자연스러운 한 장의 사진입니다.
Mosaic를 지나치게 강하게 사용하면 학습 이미지 분포가 실제 서비스 사진과 멀어질 수 있습니다.
따라서 “Mosaic가 유명하니까 무조건 사용”하지 않고 Validation 성능으로 판단합니다.

# 18. YOLO 기법 ⑨ MixUp

MixUp은 두 이미지를 반투명하게 겹치고 두 이미지의 label을 함께 사용합니다.

### 기대 효과

- 특정 배경/픽셀 패턴 암기 감소
- 작은 데이터에서 과적합 완화
- 일부 가려진 것처럼 보이는 복잡한 특징 학습

### 위험

실제 카메라 사진에서는 두 장의 장면이 반투명하게 겹치는 경우가 거의 없습니다.
따라서 높은 확률보다 낮은 확률부터 실험하는 것이 일반적으로 안전합니다.

# 19. YOLO 기법 ⑩ CutMix

CutMix는 한 이미지의 직사각형 영역을 다른 이미지에 붙여 넣습니다.

### 기대 효과

- 부분 가림(occlusion)에 대한 견고성 향상
- MixUp과 달리 붙인 영역의 실제 픽셀 강도가 유지됨

현재 Ultralytics Detection 구현은 bbox가 붙여넣기 영역에서 충분한 면적을 유지하는지 고려합니다.
하지만 우리 데이터에서 항상 좋은지는 별개이므로 실제 mAP로 판단합니다.

# 20. 이번 Detection 실험에서 Copy-Paste를 제외하는 이유

현재 공식 Ultralytics 설정에서 `copy_paste`는 polygon label이 필요한 segment/OBB 작업에 적용됩니다.

1번 노트북은 원본 polygon 일부도 **Detection용 직사각형 bbox로 변환**하여 최종 processed를 만들었습니다.
즉, 최종 데이터는 객체의 정확한 실루엣 polygon을 모두 가지고 있는 segmentation dataset이 아닙니다.

따라서 사각 bbox만 보고 객체를 잘라 붙이는 임의 Copy-Paste는 배경까지 함께 복사하여 부자연스러운 이미지를 만들 수 있으므로 이번 비교에서는 제외합니다.

# 21-A. OpenCV Control: Re-encode Only

OpenCV 증강 이미지는 실제 파일로 다시 저장해야 합니다. 저장 자체가 아주 작은 JPEG 차이를 만들 수 있으므로, **아무 픽셀 변환도 하지 않고 JPEG quality 95로 다시 저장하는 control**을 함께 측정합니다.

다른 OpenCV 실험은 최종 전체 baseline(B01)과도 비교하지만, OpenCV 내부에서는 `reencode_control`과 비교하면 “재저장 효과”와 “증강 효과”를 더 잘 구분할 수 있습니다.

# 21. OpenCV 기법 ① Brightness + Contrast

수식으로는 대략 다음과 같습니다.

```text
new_pixel = alpha × old_pixel + beta
```

- `alpha`: contrast 변화
- `beta`: 전체 밝기 이동

### 기대 효과

휴대폰 노출, 조명 세기, 역광 등으로 생기는 밝기 차이에 견고해질 수 있습니다.

이 변환은 물체 위치를 바꾸지 않으므로 bbox 좌표는 그대로입니다.

# 22. OpenCV 기법 ② Gamma Correction

Gamma는 단순히 모든 픽셀에 같은 값을 더하는 밝기 변화와 다르게
어두운 영역과 밝은 영역의 반응을 비선형적으로 바꿉니다.

### 기대 효과

- 어두운 사진의 중간 밝기 영역 변화
- 카메라/디스플레이의 비선형 밝기 차이에 대한 견고성

bbox는 변하지 않습니다.

# 23. OpenCV 기법 ③ CLAHE

CLAHE는 `Contrast Limited Adaptive Histogram Equalization`의 약자입니다.

이미지 전체에 한 번에 contrast를 적용하는 대신 작은 영역별로 contrast를 높입니다.
`Contrast Limited`가 붙는 이유는 노이즈까지 지나치게 증폭되는 것을 제한하기 때문입니다.

### 기대 효과

- 그림자 속 객체의 경계가 더 잘 보이게 만들 수 있음
- 조명이 불균일한 이미지에 대응

### 위험

원래 자연스러운 이미지보다 질감이 과도하게 강조될 수도 있습니다.

# 24. OpenCV 기법 ④ Gaussian Blur

Gaussian Blur는 주변 픽셀을 가중 평균하여 이미지를 흐리게 만듭니다.

### 어떤 상황을 모사하나?

- 초점이 살짝 맞지 않은 사진
- 손떨림보다는 렌즈 초점/해상도 저하에 가까운 blur

### 기대 효과

모델이 아주 날카로운 edge만 의존하지 않고 더 큰 형태 특징을 학습하도록 도울 수 있습니다.

# 25. OpenCV 기법 ⑤ Motion Blur

Motion Blur는 한 방향으로 픽셀을 퍼뜨려 촬영 중 카메라나 물체가 움직인 상황을 모사합니다.

### 실제 서비스와의 관계

사용자가 모바일에서 사진을 빠르게 찍으면 손 움직임으로 흐림이 생길 수 있습니다.
이런 품질 저하에 대한 robustness를 직접 실험합니다.

# 26. OpenCV 기법 ⑥ Gaussian Noise

센서 노이즈처럼 픽셀 값에 랜덤한 잡음을 더합니다.

### 기대 효과

- 저조도/작은 센서/압축 전처리에서 생기는 잡음에 대한 견고성
- 픽셀 단위 세부 패턴을 과하게 암기하는 것을 줄일 가능성

너무 강하면 객체 질감 자체를 파괴하므로 작은 sigma 범위를 사용합니다.

# 27. OpenCV 기법 ⑦ JPEG Compression Artifact

실제 웹 서비스에서는 모바일 업로드 → 네트워크 전송 → 서버 저장 과정에서 이미지가 다시 압축될 수 있습니다.

JPEG 품질을 낮췄다가 다시 디코딩하면 블록 노이즈와 ringing artifact가 생깁니다.

### 기대 효과

실제 업로드 이미지가 원본보다 압축된 경우에도 성능이 덜 떨어지도록 할 수 있습니다.

# 28. OpenCV 기법 ⑧ Affine Transform

Affine 변환은 회전, 이동, 확대/축소 등을 한 행렬로 처리합니다.

**기하학적 증강에서 가장 중요한 점은 bbox도 같은 행렬로 이동시켜야 한다는 것**입니다.

이 노트북은 bbox 네 모서리를 모두 변환하고,
변환된 네 점을 다시 감싸는 axis-aligned bbox를 계산합니다.
이미지 밖으로 너무 많이 잘린 bbox는 제거합니다.

# 29. OpenCV 기법 ⑨ Perspective Warp

이미지 네 모서리를 조금씩 움직여 원근 변형을 만듭니다.

Affine보다 더 일반적인 변환이라 직선의 평행 관계도 달라질 수 있습니다.
카메라가 물체를 사선으로 보는 상황을 흉내 냅니다.

bbox 역시 네 모서리를 homography로 변환해야 합니다.

# 30. OpenCV 기법 ⑩ Horizontal Flip

YOLO의 online flip과 같은 의미의 변환을 OpenCV로 미리 저장해 비교합니다.

이 비교를 통해 “기법 자체”뿐 아니라

- 매 epoch 랜덤하게 변하는 online augmentation
- 한번 만들어진 static offline augmentation

사이의 차이도 간접적으로 볼 수 있습니다.

# 31. OpenCV 조합 기법

단일 변환이 끝나면 서로 성격이 다른 기법을 조합합니다.

## Photometric Combo

물체 위치는 그대로 두고 밝기/대비/감마/CLAHE/노이즈/압축 품질 등을 랜덤 조합합니다.

## Geometric Combo

Affine + Perspective + Horizontal Flip을 조합합니다.

## Mixed Combo

Photometric과 Geometric을 함께 사용합니다.

### 왜 조합이 필요하나?

실제 사진에서는 “조금 어둡고 + 약간 기울어지고 + 압축된” 변화가 동시에 존재할 수 있습니다.
단, 여러 변형을 너무 많이 중첩하면 비현실적이 될 수 있으므로 조합 강도는 단일 기법보다 조금 보수적으로 설정합니다.

# 32. Ultralytics 현재 버전의 augmentation 설정 확인

Ultralytics 버전에 따라 새로운 옵션이 추가되거나 이름이 달라질 수 있습니다.

그래서 이 노트북은 설치된 버전의 `DEFAULT_CFG_DICT`를 확인하고,
지원하지 않는 실험 argument는 억지로 실행하지 않고 `SKIPPED_UNSUPPORTED`로 기록합니다.

또한 `yolo_default_baseline`은 augmentation 값을 직접 덮어쓰지 않습니다.
즉, **현재 설치된 Ultralytics 버전의 실제 기본값**이 baseline이 됩니다.

In [ ]:
AUGMENTATION_KEYS = [
    "hsv_h", "hsv_s", "hsv_v",
    "degrees", "translate", "scale", "shear", "perspective",
    "flipud", "fliplr", "bgr",
    "mosaic", "mixup", "cutmix", "copy_paste",
    "close_mosaic", "augmentations",
]

installed_aug_defaults = {
    key: DEFAULT_CFG_DICT.get(key, "<not available>")
    for key in AUGMENTATION_KEYS
}

display(
    pd.DataFrame({
        "argument": list(installed_aug_defaults.keys()),
        "installed_default": list(installed_aug_defaults.values()),
    })
)

# 33. 무증강 Control 설정

증강 기법을 한 개씩 공정하게 비교하려면 다른 증강이 동시에 켜져 있으면 안 됩니다.

예를 들어 HSV만 시험한다고 했는데 Mosaic와 flip도 기본값으로 켜져 있다면,
성능 변화가 HSV 때문인지 Mosaic 때문인지 알 수 없습니다.

그래서 `NO_AUG`를 만든 뒤 단일 기법 실험에서는 필요한 값만 다시 켭니다.

`augmentations=[]`도 지원되는 버전에서는 Albumentations 기반 추가 변형을 막는 용도로 사용합니다.

In [ ]:
NO_AUG = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "bgr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 0,
    "augmentations": [],
}


def with_no_aug(**changes):
    config = dict(NO_AUG)
    config.update(changes)
    return config


def supported_train_args(config: dict | None):
    """현재 Ultralytics 버전에서 지원되는 key만 남깁니다."""
    if config is None:
        return {}, []

    if not DEFAULT_CFG_DICT:
        # config dictionary를 가져오지 못한 버전에서는 그대로 전달합니다.
        return dict(config), []

    supported = set(DEFAULT_CFG_DICT.keys())
    unknown = sorted(set(config.keys()) - supported)
    filtered = {key: value for key, value in config.items() if key in supported}

    return filtered, unknown

# 34. YOLO 실험용 augmentation 설정

아래 값은 처음부터 “최고일 것”이라고 가정한 정답이 아닙니다.
너무 극단적이지 않은 범위의 **실험 후보**입니다.

단일 실험 → 조합 실험 순서로 비교하여 우리 데이터에 맞는 값을 찾습니다.

In [ ]:
YOLO_AUG_CONFIGS = {
    "hsv": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.50,
        hsv_v=0.35,
    ),
    "flip": with_no_aug(
        fliplr=0.50,
    ),
    "rotation": with_no_aug(
        degrees=10.0,
    ),
    "translate": with_no_aug(
        translate=0.08,
    ),
    "scale": with_no_aug(
        scale=0.25,
    ),
    "shear": with_no_aug(
        shear=2.0,
    ),
    "perspective": with_no_aug(
        perspective=0.0005,
    ),
    "mosaic": with_no_aug(
        mosaic=0.70,
        close_mosaic=10,
    ),
    "mixup": with_no_aug(
        mixup=0.15,
        close_mosaic=10,
    ),
    "cutmix": with_no_aug(
        cutmix=0.15,
        close_mosaic=10,
    ),
    "hsv_flip": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.50,
        hsv_v=0.35,
        fliplr=0.50,
    ),
    "geo_combo": with_no_aug(
        degrees=10.0,
        translate=0.08,
        scale=0.25,
        shear=2.0,
        perspective=0.0005,
        fliplr=0.50,
    ),
    "mosaic_hsv_geo": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.45,
        hsv_v=0.30,
        degrees=8.0,
        translate=0.06,
        scale=0.20,
        fliplr=0.50,
        mosaic=0.65,
        close_mosaic=10,
    ),
    "mosaic_mixup_cutmix": with_no_aug(
        mosaic=0.65,
        mixup=0.10,
        cutmix=0.10,
        close_mosaic=10,
    ),
    "balanced_combo": with_no_aug(
        hsv_h=0.012,
        hsv_s=0.40,
        hsv_v=0.30,
        degrees=7.0,
        translate=0.06,
        scale=0.18,
        shear=1.0,
        perspective=0.0003,
        fliplr=0.45,
        mosaic=0.50,
        mixup=0.05,
        cutmix=0.05,
        close_mosaic=10,
    ),
}

# 35. YOLO txt 라벨을 OpenCV용 pixel bbox로 읽기

OpenCV 기하 증강을 하려면 YOLO normalized 좌표를 pixel `x1,y1,x2,y2`로 다시 바꿔야 합니다.

변환 후 새 bbox를 다시 normalized YOLO 형식으로 저장합니다.

In [ ]:
def image_to_label_path(image_path: Path) -> Path:
    parts = list(image_path.parts)
    indices = [index for index, part in enumerate(parts) if part.lower() == "images"]

    if not indices:
        raise ValueError(f"images 폴더가 경로에 없습니다: {image_path}")

    parts[indices[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")


def read_yolo_label(label_path: Path, image_width: int, image_height: int):
    classes = []
    boxes = []

    text = label_path.read_text(encoding="utf-8").strip()

    for line in text.splitlines():
        class_id, xc, yc, bw, bh = map(float, line.split())
        class_id = int(class_id)

        x1 = (xc - bw / 2) * image_width
        y1 = (yc - bh / 2) * image_height
        x2 = (xc + bw / 2) * image_width
        y2 = (yc + bh / 2) * image_height

        classes.append(class_id)
        boxes.append([x1, y1, x2, y2])

    return (
        np.asarray(classes, dtype=int),
        np.asarray(boxes, dtype=np.float32).reshape(-1, 4),
    )


def boxes_to_yolo_lines(classes, boxes, image_width: int, image_height: int):
    lines = []

    for class_id, box in zip(classes, boxes):
        x1, y1, x2, y2 = map(float, box)

        xc = ((x1 + x2) / 2) / image_width
        yc = ((y1 + y2) / 2) / image_height
        bw = (x2 - x1) / image_width
        bh = (y2 - y1) / image_height

        lines.append(
            f"{int(class_id)} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}"
        )

    return lines

# 36. 기하 변환 시 bbox 네 모서리를 함께 이동시키는 핵심 함수

bbox는 두 점 `(x1,y1)`, `(x2,y2)`만 단순 변환하면 안 됩니다.
회전이나 perspective가 들어가면 원래 직사각형이 기울어진 사각형이 되기 때문입니다.

따라서 네 모서리를 모두 변환합니다.

```text
(x1,y1)  (x2,y1)
(x1,y2)  (x2,y2)
```

그리고 변환된 네 점을 모두 포함하는 새 axis-aligned bbox를 계산합니다.

`min_visible=0.25`는 이미지 밖으로 너무 많이 잘려 원래 bbox 면적의 25%도 남지 않은 객체는 학습 label에서 제외한다는 의미입니다.

In [ ]:
def bbox_area(boxes: np.ndarray):
    if len(boxes) == 0:
        return np.empty((0,), dtype=np.float32)

    widths = np.clip(boxes[:, 2] - boxes[:, 0], 0, None)
    heights = np.clip(boxes[:, 3] - boxes[:, 1], 0, None)
    return widths * heights


def transform_boxes(
    boxes: np.ndarray,
    matrix: np.ndarray,
    image_width: int,
    image_height: int,
    min_visible: float = 0.25,
    min_size: float = 2.0,
):
    if len(boxes) == 0:
        return boxes.copy(), np.empty((0,), dtype=bool)

    corners = np.stack(
        [
            boxes[:, [0, 1]],
            boxes[:, [2, 1]],
            boxes[:, [2, 3]],
            boxes[:, [0, 3]],
        ],
        axis=1,
    ).astype(np.float32)

    points = corners.reshape(-1, 1, 2)

    if matrix.shape == (2, 3):
        transformed = cv2.transform(points, matrix).reshape(-1, 4, 2)
    else:
        transformed = cv2.perspectiveTransform(points, matrix).reshape(-1, 4, 2)

    raw_boxes = np.column_stack([
        transformed[:, :, 0].min(axis=1),
        transformed[:, :, 1].min(axis=1),
        transformed[:, :, 0].max(axis=1),
        transformed[:, :, 1].max(axis=1),
    ]).astype(np.float32)

    raw_area = bbox_area(raw_boxes)

    clipped = raw_boxes.copy()
    clipped[:, [0, 2]] = np.clip(clipped[:, [0, 2]], 0, image_width)
    clipped[:, [1, 3]] = np.clip(clipped[:, [1, 3]], 0, image_height)

    clipped_area = bbox_area(clipped)
    visible_ratio = clipped_area / np.maximum(raw_area, 1e-6)

    keep = (
        ((clipped[:, 2] - clipped[:, 0]) >= min_size)
        & ((clipped[:, 3] - clipped[:, 1]) >= min_size)
        & (visible_ratio >= min_visible)
    )

    return clipped[keep], keep

# 37. OpenCV Photometric 증강 함수

Photometric 변환은 픽셀 값만 바꾸고 물체 위치는 바꾸지 않습니다.
따라서 bbox는 그대로 반환합니다.

In [ ]:
def cv_brightness_contrast(image, boxes, classes, rng):
    alpha = float(rng.uniform(0.75, 1.25))
    beta = float(rng.uniform(-30, 30))

    output = np.clip(
        image.astype(np.float32) * alpha + beta,
        0,
        255,
    ).astype(np.uint8)

    return output, boxes.copy(), classes.copy()


def cv_gamma(image, boxes, classes, rng):
    gamma = float(rng.uniform(0.70, 1.40))
    inverse_gamma = 1.0 / gamma

    table = np.array([
        ((value / 255.0) ** inverse_gamma) * 255
        for value in np.arange(256)
    ]).astype(np.uint8)

    output = cv2.LUT(image, table)
    return output, boxes.copy(), classes.copy()


def cv_clahe(image, boxes, classes, rng):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=float(rng.uniform(1.5, 3.0)),
        tileGridSize=(8, 8),
    )

    enhanced_l = clahe.apply(l_channel)
    output = cv2.cvtColor(
        cv2.merge([enhanced_l, a_channel, b_channel]),
        cv2.COLOR_LAB2BGR,
    )

    return output, boxes.copy(), classes.copy()


def cv_gaussian_blur(image, boxes, classes, rng):
    kernel_size = int(rng.choice([3, 5]))
    output = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    return output, boxes.copy(), classes.copy()


def cv_motion_blur(image, boxes, classes, rng):
    kernel_size = int(rng.choice([5, 7]))
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    direction = int(rng.integers(0, 4))

    if direction == 0:
        kernel[kernel_size // 2, :] = 1.0
    elif direction == 1:
        kernel[:, kernel_size // 2] = 1.0
    elif direction == 2:
        np.fill_diagonal(kernel, 1.0)
    else:
        np.fill_diagonal(np.fliplr(kernel), 1.0)

    kernel /= kernel.sum()
    output = cv2.filter2D(image, -1, kernel)
    return output, boxes.copy(), classes.copy()


def cv_gaussian_noise(image, boxes, classes, rng):
    sigma = float(rng.uniform(5.0, 18.0))
    noise = rng.normal(0, sigma, size=image.shape).astype(np.float32)

    output = np.clip(
        image.astype(np.float32) + noise,
        0,
        255,
    ).astype(np.uint8)

    return output, boxes.copy(), classes.copy()


def cv_jpeg_compression(image, boxes, classes, rng):
    quality = int(rng.integers(45, 86))
    success, encoded = cv2.imencode(
        ".jpg",
        image,
        [cv2.IMWRITE_JPEG_QUALITY, quality],
    )

    if not success:
        return image.copy(), boxes.copy(), classes.copy()

    output = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    return output, boxes.copy(), classes.copy()

# 38. OpenCV Geometric 증강 함수

이 함수들은 이미지와 bbox를 동시에 변환합니다.

- Affine: 회전 + 크기 + 이동
- Perspective: 네 모서리 원근 변형
- Horizontal Flip: x 좌표를 좌우 반전

In [ ]:
def cv_affine(image, boxes, classes, rng):
    height, width = image.shape[:2]

    angle = float(rng.uniform(-12, 12))
    scale = float(rng.uniform(0.88, 1.12))
    translate_x = float(rng.uniform(-0.07, 0.07) * width)
    translate_y = float(rng.uniform(-0.07, 0.07) * height)

    matrix = cv2.getRotationMatrix2D(
        (width / 2, height / 2),
        angle,
        scale,
    )
    matrix[0, 2] += translate_x
    matrix[1, 2] += translate_y

    output = cv2.warpAffine(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    new_boxes, keep = transform_boxes(
        boxes,
        matrix,
        width,
        height,
    )

    return output, new_boxes, classes[keep]


def cv_perspective(image, boxes, classes, rng):
    height, width = image.shape[:2]
    jitter = 0.035

    source = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1],
    ], dtype=np.float32)

    destination = source.copy()
    destination[:, 0] += rng.uniform(-jitter * width, jitter * width, 4)
    destination[:, 1] += rng.uniform(-jitter * height, jitter * height, 4)

    matrix = cv2.getPerspectiveTransform(
        source,
        destination.astype(np.float32),
    )

    output = cv2.warpPerspective(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    new_boxes, keep = transform_boxes(
        boxes,
        matrix,
        width,
        height,
    )

    return output, new_boxes, classes[keep]


def cv_horizontal_flip(image, boxes, classes, rng):
    height, width = image.shape[:2]
    output = cv2.flip(image, 1)
    new_boxes = boxes.copy()

    if len(new_boxes):
        old_x1 = boxes[:, 0].copy()
        old_x2 = boxes[:, 2].copy()
        new_boxes[:, 0] = width - old_x2
        new_boxes[:, 2] = width - old_x1

    return output, new_boxes, classes.copy()

# 39. OpenCV 조합 함수

조합은 모든 변환을 무조건 연속 적용하지 않고 일부를 확률적으로 선택합니다.
이유는 너무 많은 변환을 중첩해 현실에서 보기 어려운 이미지를 만드는 것을 피하기 위해서입니다.

In [ ]:
def cv_reencode_control(image, boxes, classes, rng):
    """픽셀 변환 없이 OpenCV decode/re-encode 영향만 측정하는 control입니다."""
    return image.copy(), boxes.copy(), classes.copy()


def cv_photometric_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = image.copy(), boxes.copy(), classes.copy()

    if rng.random() < 0.80:
        output, new_boxes, new_classes = cv_brightness_contrast(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.35:
        output, new_boxes, new_classes = cv_gamma(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.30:
        output, new_boxes, new_classes = cv_clahe(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.25:
        output, new_boxes, new_classes = cv_gaussian_noise(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.20:
        output, new_boxes, new_classes = cv_jpeg_compression(
            output, new_boxes, new_classes, rng
        )

    return output, new_boxes, new_classes


def cv_geometric_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = image.copy(), boxes.copy(), classes.copy()

    if rng.random() < 0.80:
        output, new_boxes, new_classes = cv_affine(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.30:
        output, new_boxes, new_classes = cv_perspective(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.45:
        output, new_boxes, new_classes = cv_horizontal_flip(
            output, new_boxes, new_classes, rng
        )

    return output, new_boxes, new_classes


def cv_mixed_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = cv_photometric_combo(
        image, boxes, classes, rng
    )
    output, new_boxes, new_classes = cv_geometric_combo(
        output, new_boxes, new_classes, rng
    )
    return output, new_boxes, new_classes


CV_POLICIES = {
    "reencode_control": cv_reencode_control,
    "brightness_contrast": cv_brightness_contrast,
    "gamma": cv_gamma,
    "clahe": cv_clahe,
    "gaussian_blur": cv_gaussian_blur,
    "motion_blur": cv_motion_blur,
    "gaussian_noise": cv_gaussian_noise,
    "jpeg_compression": cv_jpeg_compression,
    "affine": cv_affine,
    "perspective": cv_perspective,
    "horizontal_flip": cv_horizontal_flip,
    "photometric_combo": cv_photometric_combo,
    "geometric_combo": cv_geometric_combo,
    "mixed_combo": cv_mixed_combo,
}

# 40. OpenCV 증강 bbox 시각화 함수

실험을 돌리기 전에 **bbox가 변환된 이미지와 같이 움직이는지 반드시 눈으로 확인**합니다.

기하 증강 코드가 틀렸다면 모델은 완전히 잘못된 정답으로 학습하므로,
이 시각 검증은 성능 실험보다 먼저 해야 합니다.

In [ ]:
def draw_boxes(image, boxes, classes):
    output = image.copy()

    for box, class_id in zip(boxes, classes):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 3)
        cv2.putText(
            output,
            str(int(class_id)),
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

    return output


sample_image_path = train_images[0]
sample_image = cv2.imread(str(sample_image_path))
sample_height, sample_width = sample_image.shape[:2]
sample_label_path = image_to_label_path(sample_image_path)
sample_classes, sample_boxes = read_yolo_label(
    sample_label_path,
    sample_width,
    sample_height,
)

print("Sample:", sample_image_path.name)
display(pd.DataFrame({
    "class_id": sample_classes,
    "class_name": [CLASS_NAMES[int(class_id)] for class_id in sample_classes],
}))

# 41. 모든 OpenCV 단일/조합 증강 미리보기

각 정책에 동일한 샘플을 사용합니다.
기법마다 seed를 다르게 주어 실제 랜덤 변환이 보이도록 합니다.

이 그림은 **성능 결과가 아니라 전처리가 의도대로 작동하는지 보는 QA**입니다.

In [ ]:
preview_items = [("original", sample_image, sample_boxes, sample_classes)]

for index, (policy_name, policy_fn) in enumerate(CV_POLICIES.items(), start=1):
    rng = np.random.default_rng(SEED + index)
    augmented_image, augmented_boxes, augmented_classes = policy_fn(
        sample_image,
        sample_boxes,
        sample_classes,
        rng,
    )

    preview_items.append(
        (policy_name, augmented_image, augmented_boxes, augmented_classes)
    )

cols = 3
rows = math.ceil(len(preview_items) / cols)

plt.figure(figsize=(18, rows * 5))

for index, (name, image, boxes, classes) in enumerate(preview_items, start=1):
    drawn = draw_boxes(image, boxes, classes)

    plt.subplot(rows, cols, index)
    plt.imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
    plt.title(name)
    plt.axis("off")

plt.tight_layout()
plt.show()

# 42. OpenCV static train dataset 생성 원칙

각 OpenCV 실험은 원본 train 이미지와 **같은 개수**의 train 이미지를 생성합니다.

각 원본 이미지마다:

1. 80% 확률로 해당 policy 적용
2. 20% 확률로 원본 유지
3. 변환된 이미지는 JPEG quality 95로 저장
4. 변환되지 않은 이미지는 원본 bytes 그대로 복사
5. Validation은 절대로 복사/변환하지 않고 원본 processed Validation을 그대로 참조

### 왜 JPEG quality 95로 저장하나?

원본 해상도가 크기 때문에 수십 개 OpenCV 실험을 모두 PNG로 저장하면 디스크 사용량과 I/O 시간이 매우 커질 수 있습니다.
따라서 변환 결과는 고품질 JPEG(95)로 저장합니다.

재저장 자체의 영향을 확인하기 위해 `reencode_control` 정책도 별도 실험합니다.
이 정책은 픽셀 변환을 하지 않고 단순히 decode → JPEG quality 95 저장만 수행하므로,
다른 OpenCV 실험이 단순 재인코딩 때문인지 실제 증강 때문인지 판단할 수 있습니다.

또한 각 OpenCV 실험이 끝나면 기본적으로 static dataset cache를 삭제하여 수십 GB가 누적되지 않게 합니다.
checkpoint와 성능 CSV는 그대로 보존됩니다.

In [ ]:
def stable_seed(*parts) -> int:
    text = "|".join(map(str, parts))
    return int(hashlib.sha1(text.encode("utf-8")).hexdigest()[:8], 16)


def build_opencv_dataset(
    policy_name: str,
    seed: int,
    apply_probability: float = CV_APPLY_PROBABILITY,
    overwrite: bool = False,
):
    if policy_name not in CV_POLICIES:
        raise KeyError(policy_name)

    policy_fn = CV_POLICIES[policy_name]
    dataset_dir = CV_CACHE_DIR / policy_name / f"seed_{seed}"
    image_dir = dataset_dir / "images" / "train"
    label_dir = dataset_dir / "labels" / "train"
    report_path = dataset_dir / "generation_report.csv"
    yaml_path = dataset_dir / "data.yaml"

    if yaml_path.exists() and report_path.exists() and not overwrite:
        return yaml_path

    if dataset_dir.exists() and overwrite:
        shutil.rmtree(dataset_dir)

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for source_image_path in train_images:
        source_label_path = image_to_label_path(source_image_path)
        image = cv2.imread(str(source_image_path))

        if image is None:
            raise ValueError(f"이미지를 읽지 못했습니다: {source_image_path}")

        height, width = image.shape[:2]
        classes, boxes = read_yolo_label(source_label_path, width, height)

        image_seed = stable_seed(seed, policy_name, source_image_path.name)
        rng = np.random.default_rng(image_seed)
        apply_aug = rng.random() < apply_probability

        if apply_aug:
            augmented_image, augmented_boxes, augmented_classes = policy_fn(
                image, boxes, classes, rng
            )

            # 기하 변환으로 원래 있던 모든 객체가 사라지면 너무 공격적인 샘플이므로 원본으로 fallback합니다.
            if len(classes) > 0 and len(augmented_classes) == 0:
                apply_aug = False
            else:
                output_image_path = image_dir / f"{source_image_path.stem}.jpg"
                success = cv2.imwrite(
                    str(output_image_path),
                    augmented_image,
                    [cv2.IMWRITE_JPEG_QUALITY, CV_OUTPUT_JPEG_QUALITY],
                )

                if not success:
                    raise IOError(f"이미지 저장 실패: {output_image_path}")

                output_label_path = label_dir / f"{source_image_path.stem}.txt"
                output_label_path.write_text(
                    "\n".join(
                        boxes_to_yolo_lines(
                            augmented_classes,
                            augmented_boxes,
                            width,
                            height,
                        )
                    ),
                    encoding="utf-8",
                )

                rows.append({
                    "source_image": source_image_path.name,
                    "output_image": output_image_path.name,
                    "augmented": True,
                    "source_objects": len(classes),
                    "output_objects": len(augmented_classes),
                })

        if not apply_aug:
            output_image_path = image_dir / source_image_path.name
            shutil.copy2(source_image_path, output_image_path)

            output_label_path = label_dir / source_label_path.name
            shutil.copy2(source_label_path, output_label_path)

            rows.append({
                "source_image": source_image_path.name,
                "output_image": output_image_path.name,
                "augmented": False,
                "source_objects": len(classes),
                "output_objects": len(classes),
            })

    generation_df = pd.DataFrame(rows)
    generation_df.to_csv(report_path, index=False, encoding="utf-8-sig")

    # train은 OpenCV static dataset, val은 원본 processed val을 그대로 사용합니다.
    cv_dataset_yaml = {
        "train": str(image_dir.resolve()),
        "val": str(VAL_IMAGE_DIR.resolve()),
        "names": {class_id: name for class_id, name in CLASS_NAMES.items()},
    }

    with open(yaml_path, "w", encoding="utf-8") as file:
        yaml.safe_dump(cv_dataset_yaml, file, allow_unicode=True, sort_keys=False)

    if len(generation_df) != len(train_images):
        raise ValueError("OpenCV dataset의 이미지 수가 원본 train과 다릅니다.")

    return yaml_path

# 43. OpenCV 기하 증강 bbox retention QA

기하 변환이 너무 강하면 bbox가 화면 밖으로 많이 잘려 객체가 삭제됩니다.

각 정책에 대해 실제 train 전체에 좌표 변환을 적용하여 다음을 확인합니다.

- 입력 객체 수
- 변환 후 유지되는 객체 수
- bbox retention 비율

retention이 지나치게 낮다면 그 증강은 물체 자체를 너무 많이 없애고 있다는 의미입니다.

In [ ]:
def bbox_retention_qa(policy_name: str, sample_limit=None):
    policy_fn = CV_POLICIES[policy_name]
    source_paths = train_images if sample_limit is None else train_images[:sample_limit]

    input_boxes = 0
    output_boxes = 0

    for image_path in source_paths:
        image = cv2.imread(str(image_path))
        height, width = image.shape[:2]
        classes, boxes = read_yolo_label(image_to_label_path(image_path), width, height)

        rng = np.random.default_rng(stable_seed("qa", policy_name, image_path.name))
        _, new_boxes, new_classes = policy_fn(image, boxes, classes, rng)

        input_boxes += len(classes)
        output_boxes += len(new_classes)

    return {
        "policy": policy_name,
        "input_objects": input_boxes,
        "output_objects": output_boxes,
        "retention_pct": 100 * output_boxes / max(input_boxes, 1),
    }


geometric_qa_policies = [
    "affine",
    "perspective",
    "horizontal_flip",
    "geometric_combo",
    "mixed_combo",
]

bbox_qa_df = pd.DataFrame([
    bbox_retention_qa(policy_name)
    for policy_name in geometric_qa_policies
])

display(bbox_qa_df)
bbox_qa_df.to_csv(
    SUMMARY_DIR / "opencv_bbox_retention_qa.csv",
    index=False,
    encoding="utf-8-sig",
)

# 44. 실험 후보 정의

실험 ID 규칙:

- `B`: Baseline / Control
- `Y`: YOLO 내장 augmentation
- `C`: OpenCV augmentation
- `H`: Hybrid (OpenCV + YOLO)

## 두 개의 Baseline이 있는 이유

### B00 `no_aug_control`

모든 제어 가능한 증강을 끈 순수 control입니다.
“증강 자체가 없는 경우”를 확인합니다.

### B01 `yolo_default_baseline`

우리가 augmentation 값을 덮어쓰지 않고 Ultralytics 기본 학습 설정을 그대로 사용합니다.
실제 프로젝트의 기본 baseline은 이 값을 기준으로 보는 것이 편리합니다.

단일 증강들은 NO_AUG에서 해당 기법만 켜므로 다른 증강과 효과가 섞이지 않습니다.

In [ ]:
EXPERIMENTS = [
    # Controls
    {
        "id": "B00", "name": "no_aug_control", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": NO_AUG, "required_args": [],
        "description": "모든 제어 가능한 augmentation OFF",
    },
    {
        "id": "B01", "name": "yolo_default_baseline", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": None, "required_args": [],
        "description": "Ultralytics 현재 버전의 기본 augmentation 그대로",
    },

    # YOLO single-factor screening
    {
        "id": "Y01", "name": "yolo_hsv", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["hsv"], "required_args": ["hsv_h", "hsv_s", "hsv_v"],
        "description": "HSV 색조/채도/밝기 변화",
    },
    {
        "id": "Y02", "name": "yolo_horizontal_flip", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["flip"], "required_args": ["fliplr"],
        "description": "좌우 반전",
    },
    {
        "id": "Y03", "name": "yolo_rotation", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["rotation"], "required_args": ["degrees"],
        "description": "약한 회전",
    },
    {
        "id": "Y04", "name": "yolo_translate", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["translate"], "required_args": ["translate"],
        "description": "객체 위치 이동",
    },
    {
        "id": "Y05", "name": "yolo_scale", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["scale"], "required_args": ["scale"],
        "description": "확대/축소",
    },
    {
        "id": "Y06", "name": "yolo_shear", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["shear"], "required_args": ["shear"],
        "description": "약한 shear 기울임",
    },
    {
        "id": "Y07", "name": "yolo_perspective", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["perspective"], "required_args": ["perspective"],
        "description": "약한 원근 왜곡",
    },
    {
        "id": "Y08", "name": "yolo_mosaic", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic"], "required_args": ["mosaic"],
        "description": "여러 이미지를 한 학습 장면에 구성",
    },
    {
        "id": "Y09", "name": "yolo_mixup", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mixup"], "required_args": ["mixup"],
        "description": "두 이미지와 label을 blending",
    },
    {
        "id": "Y10", "name": "yolo_cutmix", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["cutmix"], "required_args": ["cutmix"],
        "description": "다른 이미지의 직사각형 영역을 붙여 occlusion 생성",
    },

    # YOLO combinations
    {
        "id": "Y11", "name": "yolo_hsv_flip", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["hsv_flip"], "required_args": ["hsv_h", "fliplr"],
        "description": "HSV + 좌우 반전",
    },
    {
        "id": "Y12", "name": "yolo_geometry_combo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["geo_combo"], "required_args": ["degrees", "translate", "scale"],
        "description": "회전/이동/scale/shear/perspective/flip 조합",
    },
    {
        "id": "Y13", "name": "yolo_mosaic_hsv_geo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic_hsv_geo"], "required_args": ["mosaic", "hsv_h", "degrees"],
        "description": "Mosaic + HSV + 약한 geometry",
    },
    {
        "id": "Y14", "name": "yolo_mosaic_mixup_cutmix", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic_mixup_cutmix"], "required_args": ["mosaic", "mixup", "cutmix"],
        "description": "세 가지 multi-image augmentation 조합",
    },
    {
        "id": "Y15", "name": "yolo_balanced_combo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["balanced_combo"], "required_args": ["hsv_h", "degrees", "mosaic"],
        "description": "색/기하/multi-image를 모두 약하게 섞은 조합",
    },

    # OpenCV static augmentation
    *[
        {
            "id": f"C{index:02d}",
            "name": f"opencv_{policy_name}",
            "family": "opencv_single" if policy_name not in {"photometric_combo", "geometric_combo", "mixed_combo"} else "opencv_combo",
            "data_kind": "opencv",
            "cv_policy": policy_name,
            "aug": NO_AUG,
            "required_args": [],
            "description": f"OpenCV static augmentation: {policy_name}",
        }
        for index, policy_name in enumerate(CV_POLICIES.keys(), start=1)
    ],

    # Hybrid: static OpenCV + online YOLO
    {
        "id": "H01", "name": "cv_photo_plus_yolo_geo", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "photometric_combo",
        "aug": YOLO_AUG_CONFIGS["geo_combo"], "required_args": ["degrees", "translate", "scale"],
        "description": "OpenCV photometric + YOLO online geometry",
    },
    {
        "id": "H02", "name": "cv_geo_plus_yolo_hsv", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "geometric_combo",
        "aug": YOLO_AUG_CONFIGS["hsv"], "required_args": ["hsv_h", "hsv_s", "hsv_v"],
        "description": "OpenCV geometry + YOLO online HSV",
    },
    {
        "id": "H03", "name": "cv_mixed_plus_yolo_light", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "mixed_combo",
        "aug": with_no_aug(
            hsv_h=0.010, hsv_s=0.30, hsv_v=0.25,
            degrees=5.0, translate=0.04, scale=0.12,
            fliplr=0.30, mosaic=0.30, close_mosaic=10,
        ),
        "required_args": ["hsv_h", "degrees", "mosaic"],
        "description": "OpenCV mixed + 약한 YOLO online augmentation",
    },
    {
        "id": "H04", "name": "cv_jpeg_plus_yolo_hsv_geo", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "jpeg_compression",
        "aug": with_no_aug(
            hsv_h=0.012, hsv_s=0.40, hsv_v=0.30,
            degrees=7.0, translate=0.05, scale=0.15, fliplr=0.40,
        ),
        "required_args": ["hsv_h", "degrees", "scale"],
        "description": "업로드 압축 품질 저하 + YOLO 색/기하 변화",
    },
]

if SMOKE_TEST:
    smoke_ids = {"B00", "B01", "Y01", "C01", "H01"}
    EXPERIMENTS = [experiment for experiment in EXPERIMENTS if experiment["id"] in smoke_ids]

planned_experiments_df = pd.DataFrame([
    {
        "id": experiment["id"],
        "name": experiment["name"],
        "family": experiment["family"],
        "data_kind": experiment["data_kind"],
        "cv_policy": experiment["cv_policy"],
        "description": experiment["description"],
    }
    for experiment in EXPERIMENTS
])

display(planned_experiments_df)
print("총 본 실험 수:", len(EXPERIMENTS))

# 45. 실험 지원 여부 검사

특히 `cutmix`처럼 비교적 최근 Ultralytics 버전에서 추가된 옵션은 설치 버전이 오래되면 없을 수 있습니다.

필수 argument가 지원되지 않는 실험은 다른 설정으로 몰래 대체하지 않습니다.
대신 해당 실험을 건너뛰고 이유를 결과 CSV에 남깁니다.

In [ ]:
def missing_required_args(experiment):
    if not DEFAULT_CFG_DICT:
        return []

    supported = set(DEFAULT_CFG_DICT.keys())
    return [
        argument
        for argument in experiment.get("required_args", [])
        if argument not in supported
    ]


compatibility_rows = []

for experiment in EXPERIMENTS:
    missing = missing_required_args(experiment)
    compatibility_rows.append({
        "id": experiment["id"],
        "name": experiment["name"],
        "supported": len(missing) == 0,
        "missing_required_args": ", ".join(missing),
    })

compatibility_df = pd.DataFrame(compatibility_rows)
display(compatibility_df)

# 46. 실험별 dataset YAML 선택 함수

- YOLO-only 실험 → 원본 processed train/val 사용
- OpenCV 실험 → 해당 policy의 static train dataset 생성 후 사용
- Hybrid → OpenCV static train + YOLO online augmentation

Validation 경로는 모든 경우 동일한 원본 processed Validation입니다.

In [ ]:
def dataset_yaml_for_experiment(experiment, seed):
    if experiment["data_kind"] == "processed":
        return RUNTIME_DATA_YAML

    if experiment["data_kind"] == "opencv":
        return build_opencv_dataset(
            experiment["cv_policy"],
            seed=seed,
            apply_probability=CV_APPLY_PROBABILITY,
            overwrite=False,
        )

    raise ValueError(f"알 수 없는 data_kind: {experiment['data_kind']}")

# 47. Validation metrics 추출 함수

모든 실험에서 정확히 같은 열 이름으로 결과를 저장해야 마지막에 자동 비교가 가능합니다.

저장 지표:

- Precision
- Recall
- F1 (global Precision/Recall로 계산)
- mAP50
- mAP75
- mAP50-95
- 이미지당 inference ms
- 실제 수행 epoch
- 학습 시간

In [ ]:
def extract_metrics(metrics):
    box = metrics.box

    precision = float(getattr(box, "mp", np.nan))
    recall = float(getattr(box, "mr", np.nan))

    if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    speed = getattr(metrics, "speed", {}) or {}

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": float(box.map50),
        "mAP75": float(box.map75),
        "mAP50_95": float(box.map),
        "inference_ms_per_image": speed.get("inference", np.nan),
    }

# 48. 클래스별 AP 저장 함수

전체 mAP만 저장하면 특정 클래스가 좋아졌는지 나빠졌는지 알 수 없습니다.

그래서 **모든 성공한 실험마다 클래스별 mAP50-95 CSV**를 따로 저장합니다.
Validation 객체 수가 0인 클래스는 나중에 해석에서 제외합니다.

In [ ]:
def save_per_class_metrics(metrics, experiment_id, experiment_name, seed):
    maps = np.asarray(metrics.box.maps, dtype=float)

    per_class = pd.DataFrame({
        "class_id": range(len(maps)),
        "class_name": [CLASS_NAMES.get(index, f"class_{index}") for index in range(len(maps))],
        "mAP50_95": maps,
    })

    if len(class_support_df):
        support_cols = [col for col in ["class_name", "train", "val"] if col in class_support_df.columns]
        per_class = per_class.merge(
            class_support_df[support_cols],
            on="class_name",
            how="left",
        )

    path = PER_CLASS_DIR / f"{experiment_id}_{experiment_name}_seed{seed}.csv"
    per_class.to_csv(path, index=False, encoding="utf-8-sig")
    return path

# 49. 한 실험을 처음부터 끝까지 실행하는 함수

한 실험의 순서:

1. 지원되지 않는 argument인지 검사
2. 필요한 dataset YAML 준비
3. GPU memory 정리
4. **새 `YOLO(MODEL_NAME)` 생성**
5. 동일한 공통 학습 조건으로 train
6. 저장된 `best.pt` 로드
7. 같은 Validation에서 정식 평가
8. 핵심 metric 추출
9. 클래스별 AP 저장
10. checkpoint와 결과 경로 기록

중간에 한 실험이 실패해도 전체 검색이 멈추지 않도록 오류를 결과표에 기록합니다.

In [ ]:
RESULTS_CSV = SUMMARY_DIR / "experiment_results.csv"


def train_one_experiment(experiment, seed=SEED):
    missing = missing_required_args(experiment)

    if missing:
        return {
            "status": "SKIPPED_UNSUPPORTED",
            "id": experiment["id"],
            "name": experiment["name"],
            "family": experiment["family"],
            "seed": seed,
            "error": f"unsupported args: {missing}",
        }

    dataset_yaml = dataset_yaml_for_experiment(experiment, seed)
    run_name = f"{experiment['id']}_{experiment['name']}_seed{seed}"

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 매우 중요: 모든 실험을 같은 pretrained weight에서 새로 시작합니다.
    model = YOLO(MODEL_NAME)

    train_kwargs = {
        "data": str(dataset_yaml),
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "patience": PATIENCE,
        "device": DEVICE,
        "workers": WORKERS,
        "seed": seed,
        "deterministic": True,
        "optimizer": "auto",
        "amp": True,
        "cache": False,
        "project": str(RUNS_DIR),
        "name": run_name,
        "exist_ok": True,
        "plots": True,
        "verbose": True,
    }

    filtered_aug, unknown_aug = supported_train_args(experiment["aug"])

    if experiment["aug"] is not None:
        train_kwargs.update(filtered_aug)

    start_time = time.perf_counter()
    train_result = model.train(**train_kwargs)
    train_minutes = (time.perf_counter() - start_time) / 60.0

    save_dir = Path(train_result.save_dir)
    best_pt = save_dir / "weights" / "best.pt"

    if not best_pt.exists():
        raise FileNotFoundError(best_pt)

    history_csv = save_dir / "results.csv"
    actual_epochs = np.nan

    if history_csv.exists():
        history = pd.read_csv(history_csv)
        actual_epochs = len(history)

    best_model = YOLO(str(best_pt))

    val_metrics = best_model.val(
        data=str(dataset_yaml),
        split="val",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        plots=False,
        verbose=False,
    )

    per_class_path = save_per_class_metrics(
        val_metrics,
        experiment["id"],
        experiment["name"],
        seed,
    )

    row = {
        "status": "OK",
        "id": experiment["id"],
        "name": experiment["name"],
        "family": experiment["family"],
        "description": experiment["description"],
        "seed": seed,
        "data_kind": experiment["data_kind"],
        "cv_policy": experiment["cv_policy"],
        "dataset_yaml": str(dataset_yaml),
        "requested_epochs": EPOCHS,
        "actual_epochs": actual_epochs,
        "train_minutes": train_minutes,
        "best_pt": str(best_pt),
        "save_dir": str(save_dir),
        "per_class_csv": str(per_class_path),
        "unknown_filtered_aug_args": ",".join(unknown_aug),
        **extract_metrics(val_metrics),
    }

    # OpenCV static dataset은 용량이 클 수 있으므로 결과 보고서를 보존한 뒤 cache를 정리합니다.
    if experiment["data_kind"] == "opencv" and CLEANUP_CV_DATASET_AFTER_RUN:
        dataset_dir = Path(dataset_yaml).parent
        generation_report = dataset_dir / "generation_report.csv"

        if generation_report.exists():
            persistent_report = SUMMARY_DIR / f"opencv_generation_{experiment['cv_policy']}_seed{seed}.csv"
            shutil.copy2(generation_report, persistent_report)

        shutil.rmtree(dataset_dir, ignore_errors=True)

    del model
    del best_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row

# 50. 중간 결과를 잃지 않는 실행 관리자

수십 개 실험은 오래 걸릴 수 있습니다.
마지막 실험에서 오류가 났다고 앞의 결과까지 잃으면 안 됩니다.

따라서 각 실험이 끝날 때마다 `experiment_results.csv`를 갱신합니다.

`SKIP_COMPLETED=True`이면 같은 `experiment id + seed`가 이미 성공한 경우 다시 학습하지 않습니다.
중간에 노트북이 중단되어도 다시 실행하기 편합니다.

In [ ]:
def load_existing_results():
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame()


def has_successful_run(existing_df, experiment_id, seed):
    if not len(existing_df):
        return False

    required_cols = {"status", "id", "seed"}
    if not required_cols.issubset(existing_df.columns):
        return False

    mask = (
        existing_df["status"].eq("OK")
        & existing_df["id"].eq(experiment_id)
        & existing_df["seed"].eq(seed)
    )

    return bool(mask.any())


def run_experiment_list(experiments, seed=SEED):
    existing_df = load_existing_results()
    new_rows = []

    for index, experiment in enumerate(experiments, start=1):
        print("=" * 100)
        print(f"[{index}/{len(experiments)}] {experiment['id']} - {experiment['name']} - seed={seed}")
        print(experiment["description"])

        if SKIP_COMPLETED and has_successful_run(existing_df, experiment["id"], seed):
            print("이미 성공한 결과가 있어 건너뜁니다.")
            continue

        try:
            row = train_one_experiment(experiment, seed=seed)
        except Exception as error:
            row = {
                "status": "FAILED",
                "id": experiment["id"],
                "name": experiment["name"],
                "family": experiment["family"],
                "seed": seed,
                "error": repr(error),
            }

        new_rows.append(row)

        current_existing = load_existing_results()
        combined = pd.concat(
            [current_existing, pd.DataFrame([row])],
            ignore_index=True,
        )

        # 같은 id+seed가 여러 번 존재하면 가장 최근 행을 유지합니다.
        if {"id", "seed"}.issubset(combined.columns):
            combined = combined.drop_duplicates(
                subset=["id", "seed"],
                keep="last",
            )

        combined.to_csv(
            RESULTS_CSV,
            index=False,
            encoding="utf-8-sig",
        )

        display(pd.DataFrame([row]))

    return load_existing_results()

# 51. 전체 Baseline + 증강 후보 실행

이 셀은 실제로 가장 오래 걸리는 부분입니다.

모든 실험이 같은 조건에서 실행됩니다.

```text
same model
same processed split
same image size
same batch
same max epoch
same early stopping rule
same seed
```

OpenCV 정책만 static train 입력이 바뀌고,
YOLO 정책은 online augmentation argument만 바뀝니다.

In [ ]:
if RUN_EXPERIMENTS:
    experiment_results_df = run_experiment_list(EXPERIMENTS, seed=SEED)
else:
    print("RUN_EXPERIMENTS=False: 학습을 실행하지 않습니다.")
    experiment_results_df = load_existing_results()

if len(experiment_results_df):
    display(experiment_results_df)

# 52. 성공한 실험만 추려 mAP50-95 순위 만들기

`B01 yolo_default_baseline`을 프로젝트 baseline으로 사용하여 각 실험의 성능 차이를 계산합니다.

```text
Delta > 0  → baseline보다 향상
Delta < 0  → baseline보다 하락
```

단일 seed 결과는 랜덤성에 영향을 받을 수 있으므로 이 순위만으로 최종 조합을 확정하지 않고,
뒤에서 상위 3개를 multi-seed로 다시 검증합니다.

In [ ]:
all_results_df = load_existing_results()

if not len(all_results_df):
    raise RuntimeError("아직 실험 결과가 없습니다.")

success_df = all_results_df[all_results_df["status"].eq("OK")].copy()

if not len(success_df):
    raise RuntimeError("성공한 실험이 없습니다.")

seed42_df = success_df[success_df["seed"].eq(SEED)].copy()
seed42_df = seed42_df.sort_values("mAP50_95", ascending=False)

baseline_match = seed42_df[seed42_df["id"].eq("B01")]

if not len(baseline_match):
    raise RuntimeError("B01 yolo_default_baseline 성공 결과가 필요합니다.")

BASELINE_MAP = float(baseline_match.iloc[0]["mAP50_95"])
seed42_df["mAP50_95_delta_vs_B01"] = seed42_df["mAP50_95"] - BASELINE_MAP

RANKING_CSV = SUMMARY_DIR / "seed42_ranking.csv"
seed42_df.to_csv(RANKING_CSV, index=False, encoding="utf-8-sig")

display(
    seed42_df[
        [
            "id", "name", "family",
            "precision", "recall", "f1",
            "mAP50", "mAP75", "mAP50_95",
            "mAP50_95_delta_vs_B01",
            "actual_epochs", "train_minutes",
        ]
    ]
)

# 53. 전체 실험 mAP50-95 막대그래프

가장 먼저 전체 실험의 mAP50-95를 한눈에 비교합니다.

점선은 B01 기본 baseline입니다.

In [ ]:
plot_df = seed42_df.sort_values("mAP50_95", ascending=True)

plt.figure(figsize=(12, max(9, len(plot_df) * 0.35)))
plt.barh(plot_df["name"], plot_df["mAP50_95"])
plt.axvline(BASELINE_MAP, linestyle="--", label=f"B01 baseline = {BASELINE_MAP:.4f}")
plt.xlabel("Validation mAP50-95")
plt.ylabel("Experiment")
plt.title("All Augmentation Experiments: mAP50-95")
plt.legend()
plt.tight_layout()
plt.show()

# 54. Baseline 대비 mAP50-95 증감 그래프

절대 점수뿐 아니라 “baseline보다 얼마나 좋아졌는가”를 봅니다.

0보다 오른쪽이면 개선, 왼쪽이면 하락입니다.

In [ ]:
delta_df = seed42_df.sort_values("mAP50_95_delta_vs_B01")

plt.figure(figsize=(12, max(9, len(delta_df) * 0.35)))
plt.barh(delta_df["name"], delta_df["mAP50_95_delta_vs_B01"])
plt.axvline(0, linewidth=1)
plt.xlabel("mAP50-95 delta vs YOLO default baseline")
plt.ylabel("Experiment")
plt.title("Augmentation Gain / Loss vs Baseline")
plt.tight_layout()
plt.show()

# 55. Precision / Recall / mAP50 / mAP50-95 동시 비교

mAP50-95만 가장 높아도 Recall이 크게 떨어지는 조합이라면 서비스 목적에 따라 최종 선택이 달라질 수 있습니다.

상위 15개를 네 지표로 함께 봅니다.

In [ ]:
top_metric_df = seed42_df.head(min(15, len(seed42_df))).copy()
metrics_to_plot = ["precision", "recall", "mAP50", "mAP50_95"]

x = np.arange(len(top_metric_df))
width = 0.20

plt.figure(figsize=(16, 7))

for index, metric in enumerate(metrics_to_plot):
    offset = (index - (len(metrics_to_plot) - 1) / 2) * width
    plt.bar(x + offset, top_metric_df[metric], width=width, label=metric)

plt.xticks(x, top_metric_df["name"], rotation=60, ha="right")
plt.ylabel("Score")
plt.title("Top Experiments: Precision / Recall / mAP")
plt.legend()
plt.tight_layout()
plt.show()

# 56. Precision-Recall 위치 비교

오른쪽 위에 있을수록 Precision과 Recall이 모두 높은 실험입니다.
각 점 옆에 experiment id를 표시합니다.

In [ ]:
plt.figure(figsize=(9, 7))
plt.scatter(seed42_df["recall"], seed42_df["precision"], alpha=0.8)

for _, row in seed42_df.iterrows():
    plt.annotate(
        row["id"],
        (row["recall"], row["precision"]),
        fontsize=8,
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision vs Recall by Experiment")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# 57. 학습시간 대비 성능

성능이 0.001 좋아졌지만 학습비용이 몇 배 증가하는 조합이라면 실제 프로젝트에서 효율이 낮을 수 있습니다.

이 그래프는 학습시간과 mAP50-95를 같이 봅니다.

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(seed42_df["train_minutes"], seed42_df["mAP50_95"], alpha=0.8)

for _, row in seed42_df.iterrows():
    plt.annotate(
        row["id"],
        (row["train_minutes"], row["mAP50_95"]),
        fontsize=8,
    )

plt.xlabel("Training minutes")
plt.ylabel("Validation mAP50-95")
plt.title("Accuracy vs Training Time")
plt.tight_layout()
plt.show()

# 58. 증강 family별 평균 성능

단일 실험만 보지 않고 다음 큰 범주가 전체적으로 어땠는지도 봅니다.

- baseline
- YOLO single
- YOLO combo
- OpenCV single
- OpenCV combo
- hybrid

표본 수가 다르므로 이것만으로 최적을 고르는 통계 분석은 아니며, 큰 경향을 보는 참고 그래프입니다.

In [ ]:
family_summary = (
    seed42_df.groupby("family")
    .agg(
        experiments=("id", "count"),
        mean_mAP50_95=("mAP50_95", "mean"),
        max_mAP50_95=("mAP50_95", "max"),
        mean_recall=("recall", "mean"),
    )
    .sort_values("mean_mAP50_95", ascending=False)
)

display(family_summary)

plt.figure(figsize=(10, 5))
plt.bar(family_summary.index, family_summary["mean_mAP50_95"])
plt.ylabel("Mean mAP50-95")
plt.title("Mean Validation mAP50-95 by Augmentation Family")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# 59. 모든 지표를 색 농도로 보는 Metric Matrix

행은 실험, 열은 주요 지표입니다.
값이 높을수록 더 높은 위치의 색으로 표시됩니다.

이 표는 “mAP는 좋은데 Recall만 유독 낮다” 같은 패턴을 빠르게 찾는 데 유용합니다.

In [ ]:
matrix_df = seed42_df.set_index("id")[["precision", "recall", "mAP50", "mAP50_95"]]

plt.figure(figsize=(9, max(8, len(matrix_df) * 0.28)))
image = plt.imshow(matrix_df.to_numpy(), aspect="auto", vmin=0, vmax=1)
plt.colorbar(image, label="Score")
plt.yticks(np.arange(len(matrix_df)), matrix_df.index)
plt.xticks(np.arange(len(matrix_df.columns)), matrix_df.columns, rotation=30)
plt.title("Metric Matrix")
plt.tight_layout()
plt.show()

# 60. 상위 3개 조합을 Multi-Seed로 재검증하는 이유

딥러닝은 같은 코드라도 다음 요소 때문에 결과가 조금 흔들릴 수 있습니다.

- mini-batch 순서
- augmentation random sampling
- GPU 연산의 비결정성 일부

한 번 우연히 잘 나온 실험을 “최적”이라고 선택하는 것을 줄이기 위해
초기 seed에서 상위 3개 설정을 다음 seed로 다시 학습합니다.

```text
42, 123, 777
```

최종 선택은 **mAP50-95 평균이 높고 표준편차가 과도하게 크지 않은 조합**을 우선합니다.

In [ ]:
MULTI_SEEDS = [42, 123, 777]
TOP_K_FOR_MULTI_SEED = 3

experiment_by_id = {experiment["id"]: experiment for experiment in EXPERIMENTS}

top_ids = (
    seed42_df
    .drop_duplicates("id")
    .head(TOP_K_FOR_MULTI_SEED)["id"]
    .tolist()
)

print("Multi-seed 대상:", top_ids)

if RUN_MULTI_SEED:
    for experiment_id in top_ids:
        experiment = experiment_by_id[experiment_id]

        for seed in MULTI_SEEDS:
            # seed 42가 이미 성공했다면 SKIP_COMPLETED가 자동으로 건너뜁니다.
            run_experiment_list([experiment], seed=seed)
else:
    print("RUN_MULTI_SEED=False")

# 61. Multi-Seed 평균과 표준편차 비교

평균이 가장 높은 설정을 우선하되, 표준편차도 함께 봅니다.

예:

```text
A: 0.40 ± 0.01
B: 0.41 ± 0.08
```

B가 한 seed에서는 높더라도 변동성이 매우 크다면 A가 더 안정적인 선택일 수 있습니다.

In [ ]:
all_results_df = load_existing_results()
multiseed_raw_df = all_results_df[
    all_results_df["status"].eq("OK")
    & all_results_df["id"].isin(top_ids)
    & all_results_df["seed"].isin(MULTI_SEEDS)
].copy()

multiseed_summary_df = (
    multiseed_raw_df
    .groupby(["id", "name", "family"])
    .agg(
        runs=("mAP50_95", "count"),
        mAP50_95_mean=("mAP50_95", "mean"),
        mAP50_95_std=("mAP50_95", "std"),
        mAP50_mean=("mAP50", "mean"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        train_minutes_mean=("train_minutes", "mean"),
    )
    .reset_index()
    .sort_values(["mAP50_95_mean", "recall_mean"], ascending=[False, False])
)

MULTISEED_SUMMARY_CSV = SUMMARY_DIR / "multiseed_summary.csv"
multiseed_summary_df.to_csv(
    MULTISEED_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig",
)

display(multiseed_summary_df)

In [ ]:
if len(multiseed_summary_df):
    plot_df = multiseed_summary_df.sort_values("mAP50_95_mean", ascending=True)

    plt.figure(figsize=(10, 6))
    plt.barh(
        plot_df["name"],
        plot_df["mAP50_95_mean"],
        xerr=plot_df["mAP50_95_std"].fillna(0),
        capsize=4,
    )
    plt.xlabel("Mean Validation mAP50-95")
    plt.title("Top Configurations: Multi-Seed Mean ± Std")
    plt.tight_layout()
    plt.show()

# 62. 최종 Best 설정 선택

Multi-seed 결과가 있으면 평균 mAP50-95 1위를 선택합니다.
Multi-seed를 끈 경우에는 단일 seed 순위 1위를 사용합니다.

또한 같은 설정의 여러 seed checkpoint 중 실제 mAP50-95가 가장 높았던 checkpoint를
최종 시각화용 checkpoint로 선택합니다.

In [ ]:
if len(multiseed_summary_df):
    FINAL_BEST_ID = multiseed_summary_df.iloc[0]["id"]
else:
    FINAL_BEST_ID = seed42_df.iloc[0]["id"]

candidate_rows = load_existing_results()
candidate_rows = candidate_rows[
    candidate_rows["status"].eq("OK")
    & candidate_rows["id"].eq(FINAL_BEST_ID)
].copy()

best_checkpoint_row = candidate_rows.sort_values("mAP50_95", ascending=False).iloc[0]
FINAL_BEST_PT = Path(best_checkpoint_row["best_pt"])

print("FINAL BEST ID  :", FINAL_BEST_ID)
print("FINAL BEST NAME:", best_checkpoint_row["name"])
print("Checkpoint     :", FINAL_BEST_PT)
print("Checkpoint mAP :", best_checkpoint_row["mAP50_95"])

best_config_record = {
    "best_id": FINAL_BEST_ID,
    "best_name": best_checkpoint_row["name"],
    "checkpoint": str(FINAL_BEST_PT),
    "selection_rule": "highest multi-seed mean mAP50-95, then best checkpoint for visualization",
    "ultralytics_version": ULTRALYTICS_VERSION,
    "model_name": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
}

with open(SUMMARY_DIR / "best_augmentation_config.json", "w", encoding="utf-8") as file:
    json.dump(best_config_record, file, ensure_ascii=False, indent=2)

# 63. 최종 Best 모델을 다시 Validation하고 전체 시각자료 생성

앞의 모든 실험은 비교 속도와 저장량을 위해 별도 Validation plot을 만들지 않았습니다.

최종 best에 대해서는 `plots=True`로 다시 평가하여 다음을 생성합니다.

- PR Curve
- F1 Curve
- Precision Curve
- Recall Curve
- Confusion Matrix
- Normalized Confusion Matrix
- Validation prediction batch

In [ ]:
final_best_experiment = experiment_by_id[FINAL_BEST_ID]
final_best_dataset_yaml = dataset_yaml_for_experiment(
    final_best_experiment,
    seed=int(best_checkpoint_row["seed"]),
)

final_best_model = YOLO(str(FINAL_BEST_PT))

final_metrics = final_best_model.val(
    data=str(final_best_dataset_yaml),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    plots=True,
    project=str(FINAL_DIR),
    name="best_validation",
    exist_ok=True,
    verbose=True,
)

FINAL_VAL_DIR = Path(final_metrics.save_dir)
print("Final validation outputs:", FINAL_VAL_DIR)

# 64. 최종 Best PR Curve / Confusion Matrix 표시

## PR Curve

Precision과 Recall의 trade-off를 보여줍니다.
오른쪽 위에 가까울수록 좋습니다.

## Confusion Matrix

실제 클래스와 예측 클래스가 어떤 조합에서 자주 헷갈리는지 확인합니다.
단순 전체 mAP보다 “어떤 폐기물이 어떤 폐기물로 잘못 분류되는지”를 이해하는 데 중요합니다.

In [ ]:
def display_if_exists(folder: Path, filename: str, width=1100):
    path = folder / filename

    if path.exists():
        print(filename)
        display(IPImage(filename=str(path), width=width))
    else:
        print("Not found:", path)


for filename in [
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "val_batch0_pred.jpg",
]:
    display_if_exists(FINAL_VAL_DIR, filename)

# 65. Baseline vs Best 클래스별 성능 비교

B01 baseline과 최종 best의 클래스별 mAP50-95를 비교합니다.

Validation 객체가 0인 클래스는 비교에서 제외합니다.
그 클래스는 “모델이 못했다”가 아니라 **현재 Validation 데이터로 측정할 수 없는 클래스**이기 때문입니다.

In [ ]:
all_results_df = load_existing_results()

baseline_row = (
    all_results_df[
        all_results_df["status"].eq("OK")
        & all_results_df["id"].eq("B01")
        & all_results_df["seed"].eq(SEED)
    ]
    .sort_values("mAP50_95", ascending=False)
    .iloc[0]
)

baseline_per_class = pd.read_csv(baseline_row["per_class_csv"])
best_per_class = pd.read_csv(best_checkpoint_row["per_class_csv"])

comparison_per_class = baseline_per_class[["class_id", "class_name", "mAP50_95"]].rename(
    columns={"mAP50_95": "baseline_mAP50_95"}
).merge(
    best_per_class[["class_id", "class_name", "mAP50_95"]].rename(
        columns={"mAP50_95": "best_mAP50_95"}
    ),
    on=["class_id", "class_name"],
    how="outer",
)

if len(class_support_df):
    comparison_per_class = comparison_per_class.merge(
        class_support_df[["class_name", "train", "val"]],
        on="class_name",
        how="left",
    )

comparison_per_class["delta"] = (
    comparison_per_class["best_mAP50_95"]
    - comparison_per_class["baseline_mAP50_95"]
)

measurable_class_df = comparison_per_class.copy()

if "val" in measurable_class_df.columns:
    measurable_class_df = measurable_class_df[measurable_class_df["val"] > 0]

measurable_class_df = measurable_class_df.sort_values("delta", ascending=False)

PER_CLASS_COMPARISON_CSV = SUMMARY_DIR / "baseline_vs_best_per_class.csv"
comparison_per_class.to_csv(
    PER_CLASS_COMPARISON_CSV,
    index=False,
    encoding="utf-8-sig",
)

display(measurable_class_df)

# 66. 클래스별 가장 많이 개선된 / 떨어진 항목

증강이 전체 평균을 올려도 일부 클래스는 나빠질 수 있습니다.

따라서:

- 가장 개선된 15개
- 가장 떨어진 15개

를 따로 봅니다.

In [ ]:
TOP_CLASS_N = min(15, len(measurable_class_df))

most_improved = measurable_class_df.head(TOP_CLASS_N).sort_values("delta")
most_degraded = measurable_class_df.tail(TOP_CLASS_N).sort_values("delta")

plt.figure(figsize=(11, 7))
plt.barh(most_improved["class_name"], most_improved["delta"])
plt.axvline(0, linewidth=1)
plt.xlabel("Best - Baseline mAP50-95")
plt.title("Classes Improved Most by Best Augmentation")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 7))
plt.barh(most_degraded["class_name"], most_degraded["delta"])
plt.axvline(0, linewidth=1)
plt.xlabel("Best - Baseline mAP50-95")
plt.title("Classes Degraded Most by Best Augmentation")
plt.tight_layout()
plt.show()

# 67. 실제 Validation 이미지 예측 결과 확인

정량 지표가 좋아도 실제 예측 bbox가 이상할 수 있습니다.

Validation 이미지 몇 장을 직접 추론하여 다음을 봅니다.

- 객체를 놓치지 않는가?
- bbox가 물체를 정확히 감싸는가?
- 배경을 물체로 잘못 검출하지 않는가?
- 비슷한 재활용품 클래스를 헷갈리지 않는가?

In [ ]:
NUM_PREDICTION_SAMPLES = min(12, len(val_images))
prediction_source = [str(path) for path in val_images[:NUM_PREDICTION_SAMPLES]]

prediction_results = final_best_model.predict(
    source=prediction_source,
    imgsz=IMGSZ,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(FINAL_DIR),
    name="prediction_samples",
    exist_ok=True,
    verbose=False,
)

for result in prediction_results:
    plotted_bgr = result.plot()
    plotted_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 7))
    plt.imshow(plotted_rgb)
    plt.title(Path(result.path).name)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# 68. 최종 결과 요약표 생성

팀 공유/발표용 자료는 요청한 report 경로 아래에 정리됩니다.

```text
../models/yolo/01_experiment_augmentation/report/
├── preprocess/
│   ├── class_support_processed.csv
│   ├── preprocess_summary.csv
│   └── ...
├── summary/
│   ├── experiment_results.csv
│   ├── seed42_ranking.csv
│   ├── multiseed_summary.csv
│   ├── best_augmentation_config.json
│   ├── baseline_vs_best_per_class.csv
│   └── opencv_bbox_retention_qa.csv
├── per_class/
│   └── 각 실험별 클래스 mAP CSV
└── final_best/
    └── 최종 모델 Validation 시각화와 prediction 결과
```

`experiment_results.csv`에는 모든 실험의 성능, 학습시간, checkpoint 경로가 들어 있습니다.
따라서 이 파일 하나만 읽어도 baseline부터 각 증강 조합까지 전체 결과를 다시 비교할 수 있습니다.


In [ ]:
final_summary = {
    "baseline_id": "B01",
    "baseline_name": baseline_row["name"],
    "baseline_precision": float(baseline_row["precision"]),
    "baseline_recall": float(baseline_row["recall"]),
    "baseline_mAP50": float(baseline_row["mAP50"]),
    "baseline_mAP50_95": float(baseline_row["mAP50_95"]),
    "best_id": FINAL_BEST_ID,
    "best_name": best_checkpoint_row["name"],
    "best_checkpoint_seed": int(best_checkpoint_row["seed"]),
    "best_checkpoint_precision": float(best_checkpoint_row["precision"]),
    "best_checkpoint_recall": float(best_checkpoint_row["recall"]),
    "best_checkpoint_mAP50": float(best_checkpoint_row["mAP50"]),
    "best_checkpoint_mAP50_95": float(best_checkpoint_row["mAP50_95"]),
    "mAP50_95_improvement": float(best_checkpoint_row["mAP50_95"] - baseline_row["mAP50_95"]),
}

final_summary_df = pd.DataFrame([final_summary])
final_summary_df.to_csv(
    SUMMARY_DIR / "final_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(final_summary_df)

# 69. 결과를 해석할 때 꼭 기억할 점

## 1) “증강을 많이 넣을수록 좋다”는 규칙은 없습니다.

너무 강한 Mosaic, MixUp, Perspective 등은 실제 서비스 이미지와 다른 분포를 만들 수 있습니다.
Validation 결과가 나쁘면 유명한 기법이라도 우리 데이터에서는 사용하지 않는 것이 맞습니다.

## 2) mAP50-95만 보지 말고 Recall도 봅니다.

우리 서비스에서 bbox 후보를 먼저 보여주는 UX라면 실제 객체를 놓치는 것이 큰 문제입니다.

## 3) Validation support가 0인 클래스는 평가할 수 없습니다.

현재 processed 보고서에서 Validation 객체가 없는 클래스는 클래스별 AP 해석에서 제외해야 합니다.

## 4) 모든 원본이 주간 촬영이라는 편향을 기억합니다.

Brightness/contrast 증강이 저조도 robustness에 도움을 줄 수는 있지만 실제 야간 사진을 완전히 대신하지는 못합니다.
실서비스에서 야간 입력이 예상되면 실제 야간 데이터를 추가하는 것이 가장 확실합니다.

## 5) 이 노트북의 최적 설정은 Validation 기준입니다.

증강 설정을 Validation을 보면서 골랐기 때문에 Validation 자체도 모델 선택에 사용된 데이터입니다.
최종 발표에서 “완전히 독립적인 최종 성능”을 주장하려면 별도의 Test set을 확보하는 것이 좋습니다.

# 70. 실험 완료 체크리스트

- [ ] B00 무증강 control이 성공했다.
- [ ] B01 YOLO default baseline이 성공했다.
- [ ] YOLO 단일 증강 각각의 성능이 기록되었다.
- [ ] YOLO 조합 증강 성능이 기록되었다.
- [ ] OpenCV bbox retention QA를 확인했다.
- [ ] OpenCV 단일 증강 각각의 성능이 기록되었다.
- [ ] OpenCV 조합 증강 성능이 기록되었다.
- [ ] Hybrid 실험 성능이 기록되었다.
- [ ] 모든 실험 mAP50-95 비교 그래프를 확인했다.
- [ ] Precision/Recall도 같이 확인했다.
- [ ] 상위 3개를 multi-seed로 재검증했다.
- [ ] 최종 best의 PR Curve를 확인했다.
- [ ] 최종 best의 Confusion Matrix를 확인했다.
- [ ] Baseline vs Best 클래스별 AP 변화를 확인했다.
- [ ] 실제 Validation prediction을 눈으로 확인했다.
- [ ] `best_augmentation_config.json`을 보관했다.

이 과정을 완료하면 “어떤 이미지 증강을 썼다”가 아니라,
**각 증강을 개별/조합으로 실험하고 동일한 지표로 비교하여 최종 조합을 선택했다**는 근거를 남길 수 있습니다.